# Прогнозирование популярности песен Billboard Hot 100

## Описание задачи
В этом проекте я предсказываю популярность песни (от 0 до 100 баллов) на основе её аудио-характеристик из Spotify, а также жанра, исполнителя и года выпуска.

## Что было сделано:
1. Очистка и подготовка данных (удаление пропусков, конвертация типов).
2. Feature Engineering (One-Hot Encoding для жанров и артистов, извлечение года).
3. Обучение модели LightGBM с подбором гиперпараметров.
4. Оценка качества модели.

## Результаты:
* **MAE:** 11.75
* **R2:** 0.6064

**P.S.** Мой первый пет-проект по ML-инженерии. Да, с дипсиком в помощниках, но я рад, что я не просто вайбкодил, а разбирался, уточнял и действительно много понял. Я и до этого делал такие работы, но во время учёбы (учусь в IT-вузе) и без сильного интереса, а здесь я сам захотел начать разбираться в этом направлении и сам перешёл к действиям! При самостоятельном обучении и с помощью этой работы я понял основные этапы разработки модели от подготовки датасета до валидации, а также какие существуют библиотеки, модели и для каких задач они применяются, как данные разбиваются на тренировочные и тестовые, что такое кросс-валидация и как она работает, что такое data leakage и многое другое. Всем удачи!

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

df = pd.read_csv('/kaggle/input/datasets/luthfimuthathohirin/billboard-hot-100-spotify-audio-features/billboard_hot_100_spotify_audio_features_enriched.csv')

print("Ниже информация о датасете")
df.info()

print("первые 5 строк")
print(df.head())

# 1. Сначала заполняем пропуски в explicit значением False, чтобы не было NaN
df['spotify_track_explicit'] = df['spotify_track_explicit'].fillna(False)

# 2. Теперь безопасно переводим True/False в 1/0
df['spotify_track_explicit'] = df['spotify_track_explicit'].astype(int)

# 3. Переводим длительность в секунды
df['duration_sec'] = df['spotify_track_duration_ms'] / 1000

# 4. Оставляем нужные колонки
features_to_keep = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 
    'valence', 'tempo', 'time_signature', 'spotify_track_explicit', 
    'duration_sec', 'spotify_track_popularity'
]

df_clean = df[features_to_keep].copy()

# 5. Теперь удаляем строки, где есть пропуски в ОСТАЛЬНЫХ аудио-колонках
df_clean = df_clean.dropna()

print("После очистки осталось строк:", df_clean.shape[0])

# 6. Отделяем X и y
X = df_clean.drop('spotify_track_popularity', axis=1)
y = df_clean['spotify_track_popularity']

# #Обучение модели
# #1 Делим данные
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# #2 Собираем пайплайн
# pipeline = Pipeline([
#     ('scaler', StandardScaler()), #приводим все признаки к одному масштабу
#     ('model', LinearRegression()) #Обучаем модель
# ])

# print("Обучаем модель")
# #3 обучаем весь пайплайн сразу
# pipeline.fit(X_train, y_train)

# #4 Предсказываем
# y_pred = pipeline.predict(X_test)

# #5 Метрики
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print("\n--- РЕЗУЛЬТАТЫ (с масштабированием) ---")
# print(f"MAE: {mae:.2f}")
# print(f"R2: {r2:.4f}")

# #Теперь обучаем модель Рандом Форест
# #1 Создаём модель леса (100 деревье)
# rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# #2 Обучаем (без пайплайна со скалером, лесу он не нужен)
# rf_model.fit(X_train, y_train)

# #3 Предсказываем
# y_pred_rf = rf_model.predict(X_test)

# #4 Метрики
# mae_rf = mean_absolute_error(y_test, y_pred_rf)
# r2_rf = r2_score(y_test, y_pred_rf)

# print("\n--- РЕЗУЛЬТАТЫ (Random Forest) ---")
# print(f"MAE: {mae_rf:.2f}")
# print(f"R2: {r2_rf:.4f}")

# # Достаем важность признаков из обученной модели леса
# importances = rf_model.feature_importances_

# # Создаем табличку: название признака + его важность в %
# feature_imp_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Importance': importances
# })

# # Сортируем по убыванию важности (от самого важного к худшему)
# feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

# print("\n--- ВАЖНОСТЬ ПРИЗНАКОВ (Feature Importance) ---")
# print(feature_imp_df)

# #Обучаем LightGBM
# #1 Создаём модель
# lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1)

# #Обучаем
# lgb_model.fit(X_train, y_train)

# #Предсказываем
# y_pred_lgb = lgb_model.predict(X_test)

# #смотрим метрики
# mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
# r2_lgb = r2_score(y_test, y_pred_lgb) 

# print("\n--- РЕЗУЛЬТАТЫ (LightGBM) ---")
# print(f"MAE: {mae_lgb:.2f}")
# print(f"R2: {r2_lgb:.4f}")

print("--- ДОБАВЛЯЕМ ЖАНРЫ (Feature Engineering) ---")

# 1. Берем колонку с жанрами ТОЛЬКО ДЛЯ ТЕХ СТРОК, которые остались в X
# X.index — это индексы строк, которые выжили после dropna()
genres = df.loc[X.index, 'spotify_genre'].copy()

# 2. Чистим жанры: убираем скобки и кавычки, берем первый жанр из списка
genres = genres.astype(str).str.replace(r"[\[\]']", "", regex=True).str.split(',').str[0]

# 3. Оставляем ТОП-30 самых частых жанров
top_genres = genres.value_counts().nlargest(30).index
print(f"Топ-5 самых частых жанров: {list(top_genres[:5])}...")

# 4. Все редкие жанры (которых нет в топ-30) заменяем на 'other'
genres_cleaned = np.where(genres.isin(top_genres), genres, 'other')

# 5. Делаем One-Hot Encoding
genres_encoded = pd.get_dummies(genres_cleaned, prefix='genre', dtype=int)

# 6. Склеиваем (теперь у нас и в X, и в genres_encoded ровно 27330 строк)
X_new = pd.concat([X.reset_index(drop=True), genres_encoded.reset_index(drop=True)], axis=1)
X_new = X_new.astype(float) # Приводим всё к числам

print(f"Старое количество признаков: {X.shape[1]}")
print(f"Новое количество признаков: {X_new.shape[1]} (добавили {X_new.shape[1] - X.shape[1]} жанров)")

print("\n--- ДОБАВЛЯЕМ ИСПОЛНИТЕЛЕЙ (Feature Engineering) ---")

# 1. Берем исполнителей для выживших строк
performers = df.loc[X.index, 'performer'].copy()

# 2. Берем ТОП-50 самых частых исполнителей (кто чаще всего в чартах)
top_performers = performers.value_counts().nlargest(50).index
print(f"Топ-3 исполнителя: {list(top_performers[:3])}...")

# 3. Остальных (редких) записываем в 'other'
performers_cleaned = np.where(performers.isin(top_performers), performers, 'other')

# 4. One-Hot Encoding
performers_encoded = pd.get_dummies(performers_cleaned, prefix='artist', dtype=int)

# 5. Склеиваем с нашим X_new
X_new = pd.concat([X_new.reset_index(drop=True), performers_encoded.reset_index(drop=True)], axis=1)
X_new = X_new.astype(float)

print(f"Количество признаков после добавления артистов: {X_new.shape[1]}")

print("\n--- ДОБАВЛЯЕМ ГОД ВЫПУСКА (Feature Engineering) ---")

# 1. Берем колонку с датой (проверь, как она точно называется в df.info())
# Если колонка называется по-другому (например, 'chart_date'), замени её здесь
date_col = 'first_week'

# 2. Преобразуем в datetime и достаем год
# (Если дата уже в формате строки "2023-01-01", Pandas поймет)
years = pd.to_datetime(df.loc[X.index, date_col], errors='coerce').dt.year

# 3. Заполняем пропуски (если дата битая) медианой
years = years.fillna(years.median())

# 4. Добавляем год в нашу матрицу признаков X_new
X_new['year'] = years.values

print(f"Добавили год. Диапазон лет: {years.min()} - {years.max()}")
print(f"Теперь признаков: {X_new.shape[1]}")

#Создаём базовую модель
print("\n--- ЗАПУСКАЕМ ОБУЧЕНИЕ НА НОВЫХ ПРИЗНАКАХ ---")

# 1. ВАЖНО: Делим НОВЫЕ данные (с жанрами) на train и test
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y, test_size=0.2, random_state=42
)

# 2. Задаём сетку параметров
param_grid = {
    'n_estimators': [100],          # Только 100 деревьев (убрали 200 и 300)
    'learning_rate': [0.05],        # Только 0.05 (убрали 0.01 и 0.1)
    'num_leaves': [31, 50]          # Оставили 2 варианта
}

# 3. Создаём базовую модель
lgb_base = lgb.LGBMRegressor(random_state=42, n_jobs=2, verbose=-1)

# 4. Настраиваем GridSearchCV
grid_search = GridSearchCV(
    estimator=lgb_base, 
    param_grid=param_grid, 
    cv=3, 
    scoring='r2', 
    n_jobs=2
)

# 5. Запускаем поиск на НОВЫХ тренировочных данных (X_train_new)
print("Идет поиск параметров... (пожалуйста, подожди)")
grid_search.fit(X_train_new, y_train_new)

# 6. Смотрим лучшие параметры
print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучший R2 на кросс-валидации: {grid_search.best_score_:.4f}")

# 7. Берем лучшую модель и предсказываем на НОВОМ тесте
best_lgb = grid_search.best_estimator_
y_pred_best = best_lgb.predict(X_test_new)

# 8. Метрики
mae_best = mean_absolute_error(y_test_new, y_pred_best)
r2_best = r2_score(y_test_new, y_pred_best)

print("\n--- ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ (С жанрами и артистами) ---")
print(f"MAE: {mae_best:.2f}")
print(f"R2: {r2_best:.4f}")

Ниже информация о датасете
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32503 entries, 0 to 32502
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   song_id                    32503 non-null  object 
 1   performer                  32503 non-null  object 
 2   song                       32503 non-null  object 
 3   spotify_genre              30903 non-null  object 
 4   spotify_track_id           27397 non-null  object 
 5   spotify_track_preview_url  16763 non-null  object 
 6   spotify_track_duration_ms  27397 non-null  float64
 7   spotify_track_explicit     27397 non-null  object 
 8   spotify_track_album        27391 non-null  object 
 9   danceability               27334 non-null  float64
 10  energy                     27334 non-null  float64
 11  key                        27334 non-null  float64
 12  loudness                   27334 non-null  float64
 13  mode               

/tmp/ipykernel_16/3019821033.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['spotify_track_explicit'] = df['spotify_track_explicit'].fillna(False)


Топ-5 самых частых жанров: ['adult standards', 'dance pop', 'album rock', 'contemporary country', '']...
Старое количество признаков: 14
Новое количество признаков: 45 (добавили 31 жанров)

--- ДОБАВЛЯЕМ ИСПОЛНИТЕЛЕЙ (Feature Engineering) ---
Топ-3 исполнителя: ['Taylor Swift', 'Drake', 'Glee Cast']...
Количество признаков после добавления артистов: 96

--- ДОБАВЛЯЕМ ГОД ВЫПУСКА (Feature Engineering) ---
Добавили год. Диапазон лет: 1958.0 - 2026.0
Теперь признаков: 97

--- ЗАПУСКАЕМ ОБУЧЕНИЕ НА НОВЫХ ПРИЗНАКАХ ---
Идет поиск параметров... (пожалуйста, подожди)

Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 50}
Лучший R2 на кросс-валидации: 0.6102

--- ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ (С жанрами и артистами) ---
MAE: 11.75
R2: 0.6064
